**IN:** directory of pdfs we want to extract info from, chromadb vector databases for each of those pdfs

**OUT:** csv that follows the same format as the ERA training data

In [2]:
# Import libraries
import chromadb
from openai import OpenAI
import csv
from datetime import datetime
import re
import os
from dotenv import load_dotenv
import json
import pandas as pd
from io import StringIO


In [3]:
# Paths

PDF_PATH = "pdf_processing/pdfs"
CLEANED_PDFS_PATH = "pdf_processing/finished_data"

In [7]:
# Get pdfs

pdf_files = [f for f in os.listdir(PDF_PATH) if f.endswith('.pdf')]
paper_ids = [pdf_file.split('-')[0] for pdf_file in pdf_files]


Found 1 cleaned papers: ['bo1005-leketa-2019']


In [ ]:
#  environment

load_dotenv() # get your OPENAI_API_KEY from the .env file

pre_prompt = """ 
You are a helpful assistant. You answer questions about diet information in livestock management scientific literature. 
But you only answer based on knowledge I'm providing you. You don't use your internal knowledge and you don't make things up.
If you don't know the answer, just say: I don't know
"""

user_query = """
Return a csv table with information about all the diets in the paper given below. 
The columns should be:
A.Level.Name: The name of the diet, as called in the paper
D.Item: The name of the ingredient in the diet,
D.Type: The ingredient type (Crop Byproduct, Crop Product, Forage Plants, Supplement, and Other Ingredients)
D.Amount: The amount of the ingredient in the diet
D.Unit.Amount: The units for the amount of that ingredient in the diet
DC.Is.Dry: whether the ingredient is dry
D.Ad.lib: whether it was fed ad libitum.
Notes: if available, more information about the ingredient, such as where it was sourced, how it was processed, etc. Wrap this in double quotes.
If information is not given for a specific value, write NA. 
D.Source of Notes: which section you got the Notes from, if applicable.
Return only the table in csv format without any extra response.
"""
# Here is an example of a diet csv table: 
# Ingredient,Ingredient Type,Amount,Units,Dry,Fed Ad Libitum
# Yellow maize meal,Crop Product,22.0,%,Yes,Yes
# Eragrostis curvula,Forage Plants,18.0,%,Yes,Yes
# Leucaena leucocephala,Forage Plants,25.0,%,Yes,Yes
# Wheat bran,Crop Byproduct,8.0,%,Yes,Yes

# user_query = input("What do you want to know about this paper?\n") # user query can also come from user input

output_dfs = []
# TODO add error checking, such as if it cannot properly make a df table if the response's csv format is wrong
for PAPER_ID in paper_ids:

    print(f"Currently working on {PAPER_ID}")
    
    # get cleaned pdf text
    json_filename = ""
    for root, dirs, files in os.walk(CLEANED_PDFS_PATH):
        for file in files: 
            if f"cleaned_{PAPER_ID}".lower() in file.lower():
                json_filename = file
                break

    with open(f"{CLEANED_PDFS_PATH}/{json_filename}", 'r') as file:
        cleaned_pdf_dict = json.load(file)

    sections_to_exclude = ["references", "acknowledgements", "conflicts of interest", "conflict of interest declaration", "authors' contributions"]

    cleaned_pdf_text = ""
    for section in cleaned_pdf_dict["sections"]:
        heading = section["heading"]
        if heading and heading.lower() not in sections_to_exclude:
            cleaned_pdf_text += section["heading"] + "\n"
            content = section["content"]
            for line in content:
                cleaned_pdf_text += line + "\n"
            cleaned_pdf_text += "\n"

    # get OpenAI client
    client = OpenAI()

    # define system prompt  
    system_prompt = f"""
    {pre_prompt}
    --------------------
    The data:
    {cleaned_pdf_text}
    """

    # send query to llm
    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages = [
            {"role":"system","content":system_prompt},
            {"role":"user","content":user_query} 
        ]
    )

    # print llm response
    print("\n\n---------------------\n\n")
    response_text = response.choices[0].message.content
    print(response_text)

    # convert string output into dataframe
    # remove first and last lines of ``` from output
    first_newline_ind = response_text.find('\n')
    last_newline_ind = response_text.rfind('\n')
    csv_string = response_text[first_newline_ind+1 : last_newline_ind] 
    # convert to df
    output_df = pd.read_csv(StringIO(csv_string), sep=",")
    # add B.Code column (paper ID) 
    output_df.insert(loc=0, column="B.Code", value=PAPER_ID)

    # Function to find the section heading and an excerpt for a given note
    import string

    def normalize_text(s):
        if not isinstance(s, str):
            return ""
        s = s.lower()
        s = s.replace('\n', ' ')
        s = ''.join(ch for ch in s if ch not in string.punctuation)
        s = ' '.join(s.split())
        return s

    def find_section_for_note(note, cleaned_pdf_dict):
        if not isinstance(note, str) or note.strip() == '' or note.strip().upper() == 'NA':
            return ('NA', 'NA')

        norm_note = normalize_text(note)
        note_words = [w for w in norm_note.split() if len(w) > 2]
        if not note_words:
            return ('not_found', '')

        # Try exact substring match first
        for section in cleaned_pdf_dict.get('sections', []):
            heading = section.get('heading') or ''
            content_lines = section.get('content', [])
            section_text = heading + ' ' + ' '.join(content_lines)
            if normalize_text(norm_note) in normalize_text(section_text):
                # return heading and a short excerpt (sentence containing the match)
                for sent in re.split(r'[\.\?!]\s+', ' '.join(content_lines)):
                    if normalize_text(norm_note) in normalize_text(sent):
                        return (heading or 'unknown', sent.strip())
                return (heading or 'unknown', section_text[:300])

        # Fallback: sentence overlap heuristic
        for section in cleaned_pdf_dict.get('sections', []):
            heading = section.get('heading') or ''
            content_lines = section.get('content', [])
            section_text = ' '.join(content_lines)
            sentences = re.split(r'[\.\?!]\s+', section_text)
            for sent in sentences:
                norm_sent = normalize_text(sent)
                sent_words = set([w for w in norm_sent.split() if len(w) > 2])
                if not sent_words:
                    continue
                overlap = sum(1 for w in note_words if w in sent_words)
                if overlap / max(1, len(note_words)) >= 0.4:
                    return (heading or 'unknown', sent.strip())

        return ('not_found', '')

    # add columns for Notes source section and excerpt
    notes_sections = []
    notes_excerpts = []
    for idx, row in output_df.iterrows():
        note = row.get('Notes') if 'Notes' in output_df.columns else row.get('D.Notes') if 'D.Notes' in output_df.columns else ''
        sec, exc = find_section_for_note(note, cleaned_pdf_dict)
        notes_sections.append(sec)
        notes_excerpts.append(exc)

    output_df['D.Source of Notes'] = notes_sections
    output_df['Notes_Source_Excerpt'] = notes_excerpts

    # remove rows where D.Amount is 0
    output_df = output_df[output_df['D.Amount'] != 0]

    # add table to dictionary to aggregate them all together at the end
    output_dfs.append(output_df)

all_outputs_df = pd.concat(output_dfs, ignore_index=True)




all_outputs_df.to_csv("all_outputs.csv", index=False)# print resulting csv to output folder# print resulting csv to output folder
all_outputs_df.to_csv("all_outputs.csv", index=False)


Currently working on bo1005-leketa-2019


---------------------


```csv
A.Level.Name,D.Item,D.Type,D.Amount,D.Unit.Amount,DC.Is.Dry,D.Ad.lib,Notes,D.Source of Notes
TMRL,Yellow maize meal,Crop Product,22.0,%,Yes,Yes,"NA",Materials and Methods
TMRL,Eragrostis curvula,Crop Byproduct,18.0,%,Yes,Yes,"NA",Materials and Methods
TMRL,Leucaena leucocephala,Forage Plants,25.0,%,Yes,Yes,"Previously harvested three times, dried and milled through a 25-mm screen. Included small twigs to simulate parts of the plant that a goat would eat when browsing.",Materials and Methods
TMRL,Wheat bran,Crop Byproduct,8.0,%,Yes,Yes,"NA",Materials and Methods
TMRL,Cottonseed oil cake meal,Supplement,9.0,%,Yes,Yes,"NA",Materials and Methods
TMRL,Sunflower oil cake meal,Supplement,9.0,%,Yes,Yes,"NA",Materials and Methods
TMRL,Full fat soybean meal,Supplement,0.0,%,NA,NA,"NA",Materials and Methods
TMRL,Molasses meal,Other Ingredients,7.0,%,Yes,Yes,"NA",Materials and Methods
TMRL,Mineral mix 2,Other Ingredients,2.0,